<a href="https://colab.research.google.com/github/vcellmike/PatternsFormation/blob/main/Working/2024_08_21_XGBoost_on_ImageJ_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoModel, AutoTokenizer, get_scheduler
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from tqdm.notebook import tqdm, trange
from time import perf_counter
from PIL import Image
import pandas as pd
from google.colab import drive
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# set random seeds for repeatability
import numpy as np
import random

def set_seed(seed_val):
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)

In [ ]:
seed_val = 42
set_seed(seed_val)

In [ ]:
# !unzip "/content/drive/Shareddrives/Summer_2023_VCell_Ml/2024-05-09 New Work/Saved_variables/05-09-2024_all_images-20240509T194653Z-001.zip"
# !unzip "/content/drive/Shareddrives/Summer_2023_VCell_Ml/2024-05-09 New Work/New Image Generation/Generated Images-20240523T233205Z-001.zip"
# !unzip "/content/drive/Shareddrives/Summer_2023_VCell_Ml/2024-05-09 New Work/New Image Generation/Generated_Images_7-2-24-20240702T205017Z-001.zip"
# !unzip "/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Regular.zip"
# !unzip "/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Inverse.zip"
!unzip "/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Regular_resized.zip"
!unzip "/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Inverse_resized.zip"


Archive:  /content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Regular_resized.zip
  inflating: Regular_resized/bHLH2_OE_(green)_(0-141_thresh.tif.csv  
  inflating: Regular_resized/bHLH2_RNAi_(green)_(0-170).tif.csv  
  inflating: Regular_resized/F2_260_green)_(0-105).tif.csv  
  inflating: Regular_resized/F2_A 8_(green)_(0-100).tif.csv  
  inflating: Regular_resized/LF10_11-20-23_(green)_(0-152).tif.csv  
  inflating: Regular_resized/MLC_F1_11-20-23_(green)_(0-105).tif.csv  
  inflating: Regular_resized/mpar_rto(c-c)_(green)_(0-137).tif.csv  
  inflating: Regular_resized/mpar_rto_crispr_(green)_(0-155).tif.csv  
  inflating: Regular_resized/mpar_wt_(green)_(0-170).tif.csv  
  inflating: Regular_resized/NEGAN_crispr_(green)_(0-150).tif.csv  
  inflating: Regular_resized/RTO_rnai_high_(green)_(0-103).tif.csv  
  inflating: Regular_resized/RTO_rnai_low_(green)_(0-115).tif.csv  
  inflating: Regular_re

In [ ]:
dir_1 = os.listdir("/content/05-09-2024_all_images/")
dir_2 = os.listdir("/content/Generated Images/")
dir_3 = os.listdir("/content/Generated_Images_7-2-24/")

In [ ]:
with open("/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/2024-08-15_curr_df.pkl", 'rb') as f:
  feats_df = pickle.load(f) # deserialize using load()

feats_df = feats_df.drop([700,1159,4061], axis = "rows")
feats_df.reset_index(inplace = True, drop = True)
feats_df["class"] = np.array(feats_df["class"].astype(int) -1).astype(int)
print(np.unique(np.array(feats_df["class"]), return_counts = True))

(array([0, 1, 2, 3, 4, 5, 6]), array([1993, 2329, 2330,  122,  330,  165,   35]))


In [ ]:
with open("/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/2024-08-19_feat_array_unscaled.pkl", 'rb') as f:
  feat_array_classified = pickle.load(f) # deserialize using load()

with open("/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/2024-08-18_curr_df.pkl", 'rb') as f:
  df_classes_classified = pickle.load(f) # deserialize using load()

In [ ]:
with open("/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/2024-08-19_feats_df_narrow.pkl", 'rb') as f:
  feats_df_new = pickle.load(f) # deserialize using load()

feats_df.shape

(7304, 5)

In [ ]:
df_classes_classified.head()

,class,noise,path,seed,dir,pred_class,pc1,pc2
0,2,2,0.png,0,/content/05-09-2024_all_images/,1,-31.683971,-14.198348
1,None,2,1.png,1,/content/05-09-2024_all_images/,2,31.977729,-17.444531
2,2,2,2.png,2,/content/05-09-2024_all_images/,1,-31.683971,-14.198348
3,None,2,3.png,3,/content/05-09-2024_all_images/,2,31.347501,-16.048195
4,2,2,4.png,4,/content/05-09-2024_all_images/,1,-31.683971,-14.198348


In [ ]:
feats_df.head()

,class,noise,path,seed,dir
0,2,4,noise 4 20604.0.png,20604.0,/content/Generated_Images_7-2-24/
1,2,6,noise 6 20289.0.png,20289.0,/content/Generated_Images_7-2-24/
2,3,23782.0,noise 6 23782.0.png,23782.0,/content/Generated Images/
3,1,2,7761.png,7761,/content/05-09-2024_all_images/
4,0,2,8366.png,8366,/content/05-09-2024_all_images/


In [ ]:
feats_df["pred_class"] = feats_df["class"]

In [ ]:

df1 = df_classes_classified[["noise","path","seed","dir","pred_class"]]
df2 = feats_df[["noise","path","seed","dir","pred_class"]]

pred_and_manual_df = pd.concat([df1, df2], axis=0)

pred_and_manual_df.reset_index(inplace = True, drop = True)
feats_df_new.reset_index(inplace = True, drop = True)

In [ ]:
#add full path to each dataframe to allow matching based on unique path
# df with all params: feats_df_new
# df with predicted classes: pred_and_manual_df

fdf_full_path = []
for i in range(feats_df_new.shape[0]):
  fdf_full_path.append(feats_df_new["dir"][i] + feats_df_new["path"][i])


cdf_full_path = []
for j in range(pred_and_manual_df.shape[0]):
  cdf_full_path.append((pred_and_manual_df["dir"][j] + pred_and_manual_df["path"][j]))


feats_df_new["full_path"] = fdf_full_path
pred_and_manual_df["full_path"] = cdf_full_path

In [ ]:
indices_1 = []
indices_2 = []
errors = 0
counter = 0
for i in range(feats_df_new.shape[0]):
  #pull up path to search for
  path = pred_and_manual_df["full_path"][i]

  #find index within feats_df that matches path
  ind_df = feats_df_new[feats_df_new["full_path"] == path]

  if ind_df.shape[0] == 1:
    ind = ind_df.index[0]
    # print(ind_df.head())
    indices_1.append(ind)
    indices_2.append(i)
  else:
    # indices.append("error")
    # print("error")
    errors += 1


  if counter % 1000 == 0:
    print(counter)

  counter += 1




0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000


In [ ]:
feats_df_new.shape[0]

33770

In [ ]:
# match dfs

feats_df_new = feats_df_new.iloc[indices_1]
pred_and_manual_df = pred_and_manual_df.iloc[indices_2]
feats_df_new.reset_index(inplace = True, drop = True)
pred_and_manual_df.reset_index(inplace = True, drop = True)

In [ ]:
same = list(np.array(feats_df_new["full_path"]) == np.array(pred_and_manual_df["full_path"]))

print(same.count(True))
print(same.count(False))
print(len(same))

28818
0
28818


In [ ]:
feats_df_new["classifier_pred_class"] = pred_and_manual_df["pred_class"]

In [ ]:
#its a feats_df of constrained parameters
feats_df_new.head()

feats_df = feats_df_new

In [ ]:
feats_df.head()

,Ua,Ui,Ga,Gi,Ba,Da,Di,pattern,noise,path,...,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std,full_path,classifier_pred_class
0,0.025122,0.063822,0.071957,0.106327,-0.113131,0.009186,0.611871,1,2,1.png,...,200.0,0.0,1.001,0.0,0.999,0.0,0.979,0.0,/content/05-09-2024_all_images/1.png,2
1,0.034475,0.059956,0.080798,0.10037,-0.132577,0.008852,0.854084,1,2,3.png,...,200.0,0.0,1.000,0.0,1.000,0.0,0.981,0.0,/content/05-09-2024_all_images/3.png,2
2,0.028367,0.057912,0.073998,0.080784,-0.133789,0.008512,0.534244,1,2,7.png,...,200.0,0.0,1.000,0.0,1.000,0.0,0.982,0.0,/content/05-09-2024_all_images/7.png,2
3,0.035847,0.078762,0.07823,0.101485,-0.115752,0.010542,0.933894,1,2,12.png,...,200.0,0.0,1.000,0.0,1.000,0.0,0.969,0.0,/content/05-09-2024_all_images/12.png,2
4,0.029837,0.067296,0.079054,0.095099,-0.132865,0.009721,0.822421,1,2,13.png,...,200.0,0.0,1.000,0.0,1.000,0.0,0.968,0.0,/content/05-09-2024_all_images/13.png,2


In [ ]:
print(feats_df.columns[13:111])

Index(['num_spots', 'Mean', 'Median', 'Area_mean', 'Area_std', 'X_mean',
       'X_std', 'Y_mean', 'Y_std', 'Perim._mean', 'Perim._std', 'BX_mean',
       'BX_std', 'BY_mean', 'BY_std', 'Width_mean', 'Width_std', 'Height_mean',
       'Height_std', 'Major_mean', 'Major_std', 'Minor_mean', 'Minor_std',
       'Angle_mean', 'Angle_std', 'Circ._mean', 'Circ._std', 'Feret_mean',
       'Feret_std', 'IntDen_mean', 'IntDen_std', '%Area_mean', '%Area_std',
       'RawIntDen_mean', 'RawIntDen_std', 'FeretX_mean', 'FeretX_std',
       'FeretY_mean', 'FeretY_std', 'FeretAngle_mean', 'FeretAngle_std',
       'MinFeret_mean', 'MinFeret_std', 'AR_mean', 'AR_std', 'Round_mean',
       'Round_std', 'Solidity_mean', 'Solidity_std', 'num_spots_inverted',
       'Mean_inverted', 'Median_inverted', 'Area_inverted_mean',
       'Area_inverted_std', 'X_inverted_mean', 'X_inverted_std',
       'Y_inverted_mean', 'Y_inverted_std', 'Perim._inverted_mean',
       'Perim._inverted_std', 'BX_inverted_mean', 'BX_

In [ ]:
# feats_df = feats_df.drop([700,1159,4061], axis = "rows")
# feats_df.reset_index(inplace = True, drop = True)
# feats_df["class"] = np.array(feats_df["class"].astype(int) -1).astype(int)
# print(np.unique(np.array(feats_df["class"]), return_counts = True))

XGBOOST

In [ ]:

from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
import numpy as np
from sklearn.model_selection import RepeatedKFold
from numpy import absolute
from pandas import read_csv
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from xgboost import XGBRegressor
import openpyxl
from xgboost import cv
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import operator
# for loading/processing the images
import tensorflow
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
from keras.applications.vgg16 import preprocess_input

# models
from keras.applications.vgg16 import VGG16
from keras.models import Model

# clustering and dimension reduction
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn import tree
import graphviz
from sklearn import metrics
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from hyperopt import fmin, tpe, hp,STATUS_OK
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

In [ ]:
np.unique(feats_df["num_spots"].astype(float))

array([  0.,   1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.,
        11.,  12.,  13.,  14.,  15.,  16.,  17.,  18.,  19.,  20.,  21.,
        22.,  23.,  24.,  25.,  26.,  27.,  28.,  29.,  30.,  31.,  32.,
        33.,  34.,  35.,  36.,  37.,  38.,  39.,  40.,  41.,  42.,  43.,
        44.,  45.,  46.,  47.,  48.,  49.,  50.,  51.,  52.,  53.,  54.,
        55.,  56.,  57.,  58.,  59.,  60.,  61.,  62.,  63.,  64.,  65.,
        66.,  67.,  68.,  69.,  70.,  71.,  72.,  73.,  74.,  75.,  76.,
        77.,  78.,  79.,  80.,  81.,  82.,  83.,  84.,  85.,  86.,  87.,
        88.,  89.,  90.,  91.,  92.,  93.,  94.,  95.,  96.,  97.,  98.,
        99., 100., 101., 102., 103., 104., 105., 106., 107., 108., 109.,
       110., 111., 112., 113., 114., 115., 116., 117., 118., 119., 120.,
       121., 122., 123., 124., 125., 126., 127., 128., 129., 130., 131.,
       132., 133., 134., 135., 136., 137., 138., 139., 140., 141., 142.,
       143., 144., 145., 146., 147., 148., 149., 15

In [ ]:
# scale data

X = feats_df[feats_df.columns[13:111]].astype(float)
y = feats_df[["Ua","Ui","Ga","Gi","Da","Di","Ba"]].astype(float)

#Scale data with standardscaler
scaling=StandardScaler()

# Use fit and transform method
scaling.fit(X)
X_scaled = scaling.transform(X)

# select 20 percent for testing
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42) #create training split



# df_train = feats_df.sample(frac = 0.80, random_state = 42)
# df_train.reset_index(inplace = True, drop = True)
# df_val_test = feats_df.drop(df_train.index)

# p_out = 1

# df_val_test = df_val_test.sample(frac = p_out, random_state = 42)
# df_val = df_val_test.sample(frac = 0.5, random_state = 42)
# df_test = df_val_test.drop(df_val.index)

# df_val.reset_index(inplace = True, drop = True)
# df_test.reset_index(inplace = True, drop = True)

In [ ]:
model = xgb.XGBRegressor(n_estimators=1000, max_depth=10, eta=0.1, subsample=0.7, colsample_bytree=0.8, num_boost_round=50, objective= "reg:squarederror", device = "cuda")
model.fit(X_train, y_train)

/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [02:18:17] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "num_boost_round" } are not used.

  warnings.warn(smsg, UserWarning)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eta=0.1, eval_metric=None,
             feature_types=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=None, num_boost_round=50, ...)

In [ ]:
# make predictions
y_pred = model.predict(X_test)

/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [02:19:12] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


In [ ]:
true_vals = np.array(y_test)
print(true_vals.shape)
print(y_pred.shape)

correct = 0
incorrect = 0
p_errs = np.zeros(7)

for i in range(true_vals.shape[0]):
  pred = y_pred[i]
  true_val = true_vals[i]
  p_errs += (np.abs((pred-true_val)/true_val))






print((p_errs/true_vals.shape[0])*100)

(5764, 7)
(5764, 7)
[33.00278529 58.57642953 26.1201426  43.26177822  8.94323983 38.55970073
 34.5198971 ]


In [ ]:
# make a dataframe with the names and feat_dfs and inverse_feat_dfs:
real_imj_df = {"path":[],"feats_df":[],"inverse_feats_df":[]}

reg_paths = os.listdir("/content/Inverse_resized/")

for path in reg_paths:
  real_imj_df["path"].append(path)
  real_imj_df["feats_df"].append(pd.read_csv("/content/Regular_resized/" + path))
  real_imj_df["inverse_feats_df"].append(pd.read_csv("/content/Inverse_resized/" + path))

real_df = pd.DataFrame(real_imj_df)

In [ ]:
real_df["feats_df"][0]

,,Label,Area,Mean,StdDev,Mode,Min,Max,X,Y,...,Kurt,%Area,RawIntDen,FeretX,FeretY,FeretAngle,MinFeret,AR,Round,Solidity
0,1,bHLH2_OE_(green)_(0-141_thresh.tif,40000,67.001,112.234,0,0,255,100.000,100.000,...,-0.838,26.275,2680050,0,0,135.000,200.000,1.000,1.000,1.000
1,2,bHLH2_OE_(green)_(0-141_thresh.tif,629,255.000,0.000,255,255,255,25.869,18.262,...,NaN,100.000,160395,0,0,130.236,29.931,2.465,0.406,0.479
2,3,bHLH2_OE_(green)_(0-141_thresh.tif,34,255.000,0.000,255,255,255,68.500,2.000,...,NaN,100.000,8670,64,0,161.565,5.000,1.939,0.516,0.872
3,4,bHLH2_OE_(green)_(0-141_thresh.tif,73,255.000,0.000,255,255,255,99.856,3.253,...,NaN,100.000,18615,94,5,22.620,9.000,1.485,0.673,0.864
4,5,bHLH2_OE_(green)_(0-141_thresh.tif,550,255.000,0.000,255,255,255,130.080,29.987,...,NaN,100.000,140250,128,0,100.886,23.747,2.644,0.378,0.566
5,6,bHLH2_OE_(green)_(0-141_thresh.tif,377,255.000,0.000,255,255,255,172.792,9.720,...,NaN,100.000,96135,157,0,164.249,23.000,1.875,0.533,0.546
6,7,bHLH2_OE_(green)_(0-141_thresh.tif,12,255.000,0.000,255,255,255,196.083,0.917,...,NaN,100.000,3060,192,0,159.444,3.000,3.884,0.257,0.585
7,8,bHLH2_OE_(green)_(0-141_thresh.tif,128,255.000,0.000,255,255,255,71.633,17.148,...,NaN,100.000,32640,70,8,96.340,10.000,1.760,0.568,0.892
8,9,bHLH2_OE_(green)_(0-141_thresh.tif,1338,255.000,0.000,255,255,255,103.688,58.886,...,NaN,100.000,341190,84,10,105.051,33.470,5.638,0.177,0.448
9,10,bHLH2_OE_(green)_(0-141_thresh.tif,40,255.000,0.000,255,255,255,197.000,33.000,...,NaN,100.000,10200,195,29,116.565,6.000,1.368,0.731,0.930


In [ ]:
# Add regular features

# first list is mean, second list is std except for mean and median
feats = {'Mean':[],'Median':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
         'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
       'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
       'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
       'Round':[[],[]], 'Solidity':[[],[]]}

# feats = {'Mean_inverted':[],'Median_inverted':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
#          'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
#        'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
#        'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
#        'Round':[[],[]], 'Solidity':[[],[]]}


counter = 0

# iterate through all sims
for n in range(real_df.shape[0]): # grab the feats dataframe for this sim
  fdf = real_df["feats_df"][n] #iterate through all feats within the feats dictionary

  for i in range(len(feats.keys())): # select the current feat
    feat = list(feats.keys())[i]

    # if feat in ['Mean_inverted','Median_inverted']: # check for feats that should be added from the overall measurement (first row)
    if feat in ['Mean','Median']: # check for feats that should be added from the overall measurement (first row)
      # feats[feat].append(fdf[feat[0:-9]][0]) #inverse
      feats[feat].append(fdf[feat[:]][0])
      # feats[feat].append(fdf[feat][0])

    else:
      if fdf.shape[0] <= 1: # check if the fdf is only one measurement (empty result)
        feats[feat][0].append(np.mean(np.array(fdf[feat][0]))) # mean of sole measurement
        feats[feat][1].append(np.std(np.array(fdf[feat][0]))) #std of sole measurement (0)

      else:
        feats[feat][0].append(np.mean(np.array(fdf[feat][1:]))) # add mean for each of the rest of the feats using the remaining rows
        feats[feat][1].append(np.std(np.array(fdf[feat][1:]))) # add std for each of the rest of the feats using the remaining rows

  if counter % 1000 == 0:
    print(counter)
  counter += 1

0


In [ ]:
num_spots = []

for i in range(real_df.shape[0]):
  fdf = real_df["feats_df"][i] #iterate through all feats within the feats dictionary
  if fdf.shape[0] <= 1:
    num_spots.append(0)
  else:
    num_spots.append(fdf.shape[0]-1)

# feats_df["num_spots_inverted"] = num_spots
real_df["num_spots"] = num_spots

In [ ]:
for key in list(feats.keys()):
  # if key in ['Mean_inverted','Median_inverted']:
  if key in ['Mean','Median']:
    real_df[key] = feats[key]
  else:
    # feats_df[key + "_inverted_mean"] = feats[key][0]
    # feats_df[key + "_inverted_std"] = feats[key][1]
    real_df[key + "_mean"] = feats[key][0]
    real_df[key + "_std"] = feats[key][1]

In [ ]:
real_df.head(10)

,path,feats_df,inverse_feats_df,num_spots,Mean,Median,Area_mean,Area_std,X_mean,X_std,...,FeretAngle_mean,FeretAngle_std,MinFeret_mean,MinFeret_std,AR_mean,AR_std,Round_mean,Round_std,Solidity_mean,Solidity_std
0,bHLH2_OE_(green)_(0-141_thresh.tif.csv,Label A...,Label Are...,30,67.001,0,350.333333,436.638320,104.649567,63.380542,...,106.094367,30.422104,14.895000,11.599036,2.534433,1.161018,0.472233,0.188002,0.736767,0.158909
1,RTO_rnai_high_(green)_(0-103).tif.csv,Label Ar...,Label Ar...,11,128.246,255,1828.818182,5677.547083,113.048091,65.100510,...,121.365909,41.616273,20.817909,53.213065,1.915636,1.217189,0.704182,0.314290,0.854727,0.196756
2,mpar_rto_crispr_(green)_(0-155).tif.csv,Label Ar...,Label Ar...,2,251.322,255,19711.500000,19688.500000,148.632000,49.216000,...,135.000000,0.000000,102.700000,97.300000,1.320500,0.316500,0.803500,0.192500,0.927500,0.059500
3,F2_A 8_(green)_(0-100).tif.csv,Label Area M...,Label Area Me...,9,92.578,0,1613.555556,771.210752,114.434444,60.967917,...,88.086333,29.493138,32.186333,11.473686,3.590667,2.847227,0.372111,0.136953,0.766111,0.104342
4,NEGAN_crispr_(green)_(0-150).tif.csv,Label Area ...,Label Area ...,0,0.000,0,40000.000000,0.000000,100.000000,0.000000,...,135.000000,0.000000,200.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
5,mpar_rto(c-c)_(green)_(0-137).tif.csv,Label Ar...,Label Area...,49,23.479,0,75.163265,103.827240,99.147755,55.940428,...,98.199592,32.841924,6.559959,5.483810,2.445878,1.308504,0.527082,0.253666,0.789898,0.171918
6,mpar_wt_(green)_(0-170).tif.csv,Label Area ...,Label Area M...,66,8.077,0,19.196970,18.371767,97.896258,52.566051,...,111.274424,46.307654,3.805500,1.983291,1.763136,0.530357,0.614000,0.165891,0.844894,0.093472
7,bHLH2_RNAi_(green)_(0-170).tif.csv,Label Area M...,Label Area M...,0,0.000,0,40000.000000,0.000000,100.000000,0.000000,...,135.000000,0.000000,200.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
8,LF10_11-20-23_(green)_(0-152).tif.csv,Label ...,Label Area...,101,5.699,0,8.851485,11.066304,125.673327,57.972907,...,103.979446,47.466852,2.599564,1.875266,1.738505,0.579683,0.643267,0.218933,0.857396,0.144207
9,MLC_F1_11-20-23_(green)_(0-105).tif.csv,Label ...,Label Ar...,77,15.306,0,31.181818,46.025890,101.898026,40.868593,...,100.865364,32.826527,4.046351,3.194558,2.004753,0.920000,0.585714,0.222167,0.852494,0.123426


In [ ]:
# Add regular features

# first list is mean, second list is std except for mean and median
# feats = {'Mean':[],'Median':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
#          'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
#        'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
#        'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
#        'Round':[[],[]], 'Solidity':[[],[]]}

feats = {'Mean_inverted':[],'Median_inverted':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
         'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
       'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
       'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
       'Round':[[],[]], 'Solidity':[[],[]]}


counter = 0

# iterate through all sims
for n in range(real_df.shape[0]): # grab the feats dataframe for this sim
  fdf = real_df["inverse_feats_df"][n] #iterate through all feats within the feats dictionary

  for i in range(len(feats.keys())): # select the current feat
    feat = list(feats.keys())[i]

    if feat in ['Mean_inverted','Median_inverted']: # check for feats that should be added from the overall measurement (first row)
    # if feat in ['Mean','Median']: # check for feats that should be added from the overall measurement (first row)
      feats[feat].append(fdf[feat[0:-9]][0]) #inverse
      # feats[feat].append(fdf[feat[:]][0])

    else:
      if fdf.shape[0] <= 1: # check if the fdf is only one measurement (empty result)
        feats[feat][0].append(np.mean(np.array(fdf[feat][0]))) # mean of sole measurement
        feats[feat][1].append(np.std(np.array(fdf[feat][0]))) #std of sole measurement (0)

      else:
        feats[feat][0].append(np.mean(np.array(fdf[feat][1:]))) # add mean for each of the rest of the feats using the remaining rows
        feats[feat][1].append(np.std(np.array(fdf[feat][1:]))) # add std for each of the rest of the feats using the remaining rows

  if counter % 1000 == 0:
    print(counter)
  counter += 1

0


In [ ]:
num_spots = []

for i in range(real_df.shape[0]):
  fdf = real_df["inverse_feats_df"][i] #iterate through all feats within the feats dictionary
  if fdf.shape[0] <= 1:
    num_spots.append(0)
  else:
    num_spots.append(fdf.shape[0]-1)

real_df["num_spots_inverted"] = num_spots
# real_df["num_spots"] = num_spots

In [ ]:
for key in list(feats.keys()):
  if key in ['Mean_inverted','Median_inverted']:
  # if key in ['Mean','Median']:
    real_df[key] = feats[key]
  else:
    real_df[key + "_inverted_mean"] = feats[key][0]
    real_df[key + "_inverted_std"] = feats[key][1]
    # real_df[key + "_mean"] = feats[key][0]
    # real_df[key + "_std"] = feats[key][1]

In [ ]:
print(real_df.columns[3:101])

Index(['num_spots', 'Mean', 'Median', 'Area_mean', 'Area_std', 'X_mean',
       'X_std', 'Y_mean', 'Y_std', 'Perim._mean', 'Perim._std', 'BX_mean',
       'BX_std', 'BY_mean', 'BY_std', 'Width_mean', 'Width_std', 'Height_mean',
       'Height_std', 'Major_mean', 'Major_std', 'Minor_mean', 'Minor_std',
       'Angle_mean', 'Angle_std', 'Circ._mean', 'Circ._std', 'Feret_mean',
       'Feret_std', 'IntDen_mean', 'IntDen_std', '%Area_mean', '%Area_std',
       'RawIntDen_mean', 'RawIntDen_std', 'FeretX_mean', 'FeretX_std',
       'FeretY_mean', 'FeretY_std', 'FeretAngle_mean', 'FeretAngle_std',
       'MinFeret_mean', 'MinFeret_std', 'AR_mean', 'AR_std', 'Round_mean',
       'Round_std', 'Solidity_mean', 'Solidity_std', 'num_spots_inverted',
       'Mean_inverted', 'Median_inverted', 'Area_inverted_mean',
       'Area_inverted_std', 'X_inverted_mean', 'X_inverted_std',
       'Y_inverted_mean', 'Y_inverted_std', 'Perim._inverted_mean',
       'Perim._inverted_std', 'BX_inverted_mean', 'BX_

In [ ]:
# select features of real images
real_feats = real_df[real_df.columns[3:101]]

#scale data
scaling=StandardScaler()

# Use fit and transform method
scaling.fit(real_feats)
real_feats_scaled = scaling.transform(real_feats)



# predict params of images from real feats
y_pred = model.predict(real_feats_scaled)

real_df["pred_params"] = list(y_pred)

In [ ]:
real_df["pred_params"][0]

array([ 0.02974807,  0.06299408,  0.11420467,  0.07545009,  0.01008271,
        0.7894119 , -0.10632307], dtype=float32)

In [ ]:
real_df["pred_params"][0]

array([ 0.02974807,  0.06299408,  0.11420467,  0.07545009,  0.01008271,
        0.7894119 , -0.10632307], dtype=float32)

In [ ]:
for i in range(real_df.shape[0]):
  print(real_df["path"][i])
  print(real_df["pred_params"][i])

bHLH2_OE_(green)_(0-141_thresh.tif.csv
[ 0.02974807  0.06299408  0.11420467  0.07545009  0.01008271  0.7894119
 -0.10632307]
RTO_rnai_high_(green)_(0-103).tif.csv
[ 0.03545561  0.12719852  0.12560101  0.08008096  0.00921823  0.95908463
 -0.12990025]
mpar_rto_crispr_(green)_(0-155).tif.csv
[ 0.02258218  0.11371219  0.1193148   0.09445498  0.01014515  0.94286376
 -0.0993465 ]
F2_A 8_(green)_(0-100).tif.csv
[ 0.02713457  0.0611859   0.12244835  0.06386362  0.0098374   0.9292788
 -0.09699949]
NEGAN_crispr_(green)_(0-150).tif.csv
[ 0.02982357  0.05438843  0.081828    0.12095544  0.01028309  0.8808678
 -0.08770181]
mpar_rto(c-c)_(green)_(0-137).tif.csv
[ 0.02763052  0.12534462  0.10830157  0.10573891  0.0104164   0.7709196
 -0.10005046]
mpar_wt_(green)_(0-170).tif.csv
[ 0.02768797  0.1047422   0.09191404  0.10882644  0.01024797  0.7978362
 -0.11208577]
bHLH2_RNAi_(green)_(0-170).tif.csv
[ 0.02982357  0.05438843  0.081828    0.12095544  0.01028309  0.8808678
 -0.08770181]
LF10_11-20-23_(green